# 02 · Temperature and HDX bins

Notebook per a overlap visual, error bars i estratificació per condicions com sol o vent.

In [ ]:
from pathlib import Path
import pandas as pd

from create_groups_comf import *
from create_groups_sens import add_outcomes_group_sensation

from utils_plots import plot_bin_overlap_with_errorbars,make_binned_overlap_table
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = Path("../../data")

votes = pd.read_csv(DATA_DIR / "all_surveys(votes).csv")
stops = pd.read_csv(DATA_DIR / "all_surveys(stops).csv")
ids = pd.read_csv(DATA_DIR / "all_surveys(ID).csv")

df = votes.merge(stops, on="space_code", how="left", suffixes=("", "_stop"))
print(df.shape)
df.head()

#df = add_outcomes_grups(df)
df = add_outcomes_group_sensation(df)


In [ ]:
overlap = make_binned_overlap_table(
    df,
    value_col="<T-T_fixed+<T>>",
    outcome_col="thermal_comfort",
    bins=5,
    strategy="quantile"
)
overlap.head()

In [ ]:
plot_bin_overlap_with_errorbars(
    overlap,
    outcome_col="thermal_comfort",
    z=1.0,
    ylim=(0,0.4),
    title="Thermal comfort across temperature bins",
    output_path="figures_sens/temp_overlap_thermal_sensation_errorbars.png"
)

In [ ]:
import pandas as pd

# =========================================================
# 1. Confidence intervals for within-bin proportions
# =========================================================

def wilson_ci(k, n, z=1.96):
    if n == 0 or pd.isna(n):
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt((p * (1 - p) / n) + z**2 / (4 * n**2)) / denom
    return center - half, center + half


def add_binomial_ci_to_overlap(overlap_df, z=1.96):
    """
    Expects overlap_df from make_binned_overlap_table(...)
    Adds Wilson CI columns for the within-bin class proportions.
    """
    df = overlap_df.copy()

    ci = df.apply(
        lambda r: wilson_ci(r["count"], r["bin_n"], z=z),
        axis=1,
        result_type="expand"
    )
    df["ci_low"] = ci[0]
    df["ci_high"] = ci[1]
    df["ci_width"] = df["ci_high"] - df["ci_low"]
    return df


# =========================================================
# 2. Pairwise overlap of error bars inside each bin
# =========================================================

def compute_pairwise_errorbar_overlap(overlap_with_ci_df, outcome_col):
    """
    For each bin, compares all pairs of classes and computes:
    - absolute difference in proportions
    - CI overlap length
    - normalized overlap
    """
    df = overlap_with_ci_df.copy()
    rows = []

    for b, g in df.groupby("bin", observed=True):
        g = g.sort_values("prop", ascending=False).reset_index(drop=True)

        for i in range(len(g)):
            for j in range(i + 1, len(g)):
                a = g.loc[i]
                c = g.loc[j]

                overlap = max(0.0, min(a["ci_high"], c["ci_high"]) - max(a["ci_low"], c["ci_low"]))
                min_width = min(a["ci_width"], c["ci_width"])
                norm_overlap = overlap / min_width if (pd.notna(min_width) and min_width > 0) else np.nan

                rows.append({
                    "bin": b,
                    "bin_mean": a["bin_mean"],
                    "bin_n": a["bin_n"],
                    "class_a": a[outcome_col],
                    "class_b": c[outcome_col],
                    "prop_a": a["prop"],
                    "prop_b": c["prop"],
                    "abs_prop_diff": abs(a["prop"] - c["prop"]),
                    "ci_overlap_length": overlap,
                    "ci_overlap_norm": norm_overlap,
                })

    return pd.DataFrame(rows)


# =========================================================
# 3. Bin imbalance / concentration / entropy
# =========================================================

def compute_bin_imbalance(overlap_df, outcome_col):
    """
    One row per bin with:
    - dominant class
    - dominant share
    - min/max proportions
    - range of proportions
    - normalized entropy
    - concentration index
    - L1 distance to uniform
    """
    df = overlap_df.copy()
    out = []

    for b, g in df.groupby("bin", observed=True):
        g = g.sort_values("prop", ascending=False).reset_index(drop=True)
        p = g["prop"].values.astype(float)
        p_pos = p[p > 0]
        K = len(g)

        entropy = -np.sum(p_pos * np.log(p_pos)) if len(p_pos) else np.nan
        max_entropy = np.log(K) if K > 1 else np.nan
        entropy_norm = entropy / max_entropy if pd.notna(max_entropy) and max_entropy > 0 else np.nan

        out.append({
            "bin": b,
            "bin_mean": g["bin_mean"].iloc[0],
            "bin_min": g["bin_min"].iloc[0],
            "bin_max": g["bin_max"].iloc[0],
            "bin_median": g["bin_median"].iloc[0],
            "bin_n": g["bin_n"].iloc[0],
            "n_classes_present": K,
            "dominant_class": g[outcome_col].iloc[0],
            "dominant_prop": g["prop"].iloc[0],
            "smallest_prop": g["prop"].iloc[-1],
            "prop_range": g["prop"].iloc[0] - g["prop"].iloc[-1],
            "entropy_norm": entropy_norm,
            "concentration_index": np.sum(p**2),
            "imbalance_l1_uniform": np.sum(np.abs(p - 1 / K)),
        })

    return pd.DataFrame(out).sort_values("bin_mean").reset_index(drop=True)


# =========================================================
# 4. Dominance summary by bin
# =========================================================

def compute_bin_dominance(overlap_df, outcome_col):
    """
    Returns one row per bin with the dominant class and its proportion.
    """
    df = overlap_df.copy()
    dominant = (
        df.sort_values(["bin_mean", "prop"], ascending=[True, False])
          .groupby("bin", as_index=False)
          .first()
    )
    return dominant[[
        "bin", "bin_mean", "bin_min", "bin_max", "bin_median",
        "bin_n", outcome_col, "count", "prop"
    ]].rename(columns={
        outcome_col: "dominant_class",
        "count": "dominant_count",
        "prop": "dominant_prop"
    })


# =========================================================
# 5. One wrapper for convenience
# =========================================================

def summarize_binned_outcome(df, value_col, outcome_col, bins=10, strategy="quantile", z=1.96):
    """
    Full pipeline starting from your existing make_binned_overlap_table().
    Returns:
    - overlap_df: raw counts/props/bin summary
    - overlap_ci_df: adds CIs
    - pairwise_df: pairwise CI overlaps within bins
    - imbalance_df: entropy/concentration/imbalance by bin
    - dominance_df: dominant class by bin
    """
    overlap_df = make_binned_overlap_table(
        df=df,
        value_col=value_col,
        outcome_col=outcome_col,
        bins=bins,
        strategy=strategy,
        dropna=True
    )

    overlap_ci_df = add_binomial_ci_to_overlap(overlap_df, z=z)
    pairwise_df = compute_pairwise_errorbar_overlap(overlap_ci_df, outcome_col=outcome_col)
    imbalance_df = compute_bin_imbalance(overlap_ci_df, outcome_col=outcome_col)
    dominance_df = compute_bin_dominance(overlap_ci_df, outcome_col=outcome_col)

    return overlap_df, overlap_ci_df, pairwise_df, imbalance_df, dominance_df

# Anem a probar per les combinacions que hem obtingut

In [ ]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Output folders
# --------------------------------------------------
FIG_DIR = Path("figures_sens")
CSV_DIR = Path("csvs_sens")
FIG_DIR.mkdir(exist_ok=True)
CSV_DIR.mkdir(exist_ok=True)

# --------------------------------------------------
# 2. Nice labels for titles
# --------------------------------------------------
OUTCOME_LABELS = {
    "comfort3": "Comfort 3 (baseline)",
    "comfort3_option1": "Comfort 3 - Option 1",
    "comfort3_option2": "Comfort 3 - Option 2",
    "comfort3_UNoption": "Comfort 3 - Uncomf + vuncomf option",
    "comfort4_soft": "Comfort 4 soft (baseline)",
    "comfort4_option1": "Comfort 4 - Option 1",
    "comfort4_option2": "Comfort 4 - Option 2",
}

groups = [
    "comfort3",
    "comfort3_option1",
    "comfort3_option2",
    "comfort3_UNoption",
    "comfort4_soft",
    "comfort4_option1",
    "comfort4_option2",
]

OUTCOME_LABELS_SENSATION = {
    "sens7": "Sensation 7",
    "sens3": "Sensation 3 (baseline)",
    "sens3_option1": "Sensation 3 - Option 1",
    "sens3_option2": "Sensation 3 - Option 2",
    "sens3_option3": "Sensation 3 - Option 3",
    "sens4_option1": "Sensation 4 - Option 1",
    "sens4_option2": "Sensation 4 - Option 2",
}

groups_sensation = [
    "sens7",
    "sens3",
    "sens3_option1",
    "sens3_option2",
    "sens3_option3",
    "sens4_option1",
    "sens4_option2",
]
# --------------------------------------------------
# 3. Run summaries, plots, and save outputs
# --------------------------------------------------
all_bin_imbalance = []
all_bin_dominance = []
all_pairwise_overlap = []

for outcome in groups_sensation:
    pretty_name = OUTCOME_LABELS_SENSATION.get(outcome, outcome)

    # ---- Compute tables
    overlap_df, overlap_ci_df, pairwise_df, imbalance_df, dominance_df = summarize_binned_outcome(
        df=df,
        value_col="<T-T_fixed+<T>>",
        outcome_col=outcome,
        bins=5,
        strategy="quantile",
        z=1.96
    )

    # ---- Add outcome name to exported summary tables
    pairwise_df["outcome"] = outcome
    imbalance_df["outcome"] = outcome
    dominance_df["outcome"] = outcome

    # ---- Save all CSVs for this outcome
    overlap_df.to_csv(CSV_DIR / f"{outcome}_temp_overlap.csv", index=False)
    overlap_ci_df.to_csv(CSV_DIR / f"{outcome}_temp_overlap_ci.csv", index=False)
    pairwise_df.to_csv(CSV_DIR / f"{outcome}_temp_pairwise_overlap.csv", index=False)
    imbalance_df.to_csv(CSV_DIR / f"{outcome}_temp_bin_imbalance.csv", index=False)
    dominance_df.to_csv(CSV_DIR / f"{outcome}_temp_bin_dominance.csv", index=False)

    # ---- Plot with aligned title and unique filename
    plot_bin_overlap_with_errorbars(
        overlap_df,
        outcome_col=outcome,
        z=1.0,
        ylim=(0, 1),
        title=f"{pretty_name} across temperature bins",
        output_path=str(FIG_DIR / f"{outcome}_temp_overlap_errorbars.png")
    )


    # ---- Keep combined tables for later comparison
    all_pairwise_overlap.append(pairwise_df)
    all_bin_imbalance.append(imbalance_df)
    all_bin_dominance.append(dominance_df)

# --------------------------------------------------
# 4. Combined comparison tables across all outcomes
# --------------------------------------------------
pairwise_all_df = pd.concat(all_pairwise_overlap, ignore_index=True)
imbalance_all_df = pd.concat(all_bin_imbalance, ignore_index=True)
dominance_all_df = pd.concat(all_bin_dominance, ignore_index=True)

pairwise_all_df.to_csv(CSV_DIR / "sens_ALL_temp_pairwise_overlap.csv", index=False)
imbalance_all_df.to_csv(CSV_DIR / "sens_ALL_temp_bin_imbalance.csv", index=False)
dominance_all_df.to_csv(CSV_DIR / "sens_ALL_temp_bin_dominance.csv", index=False)

print("Done. Saved individual and combined CSVs/figures.")

# Let's see how this categories behave 

In [ ]:
pairwise_summary = (
    pairwise_all_df
    .groupby("outcome", as_index=False)
    .agg(
        mean_abs_prop_diff=("abs_prop_diff", "mean"),
        mean_ci_overlap=("ci_overlap_length", "mean"),
        mean_ci_overlap_norm=("ci_overlap_norm", "mean"),
        max_abs_prop_diff=("abs_prop_diff", "max"),
        min_ci_overlap=("ci_overlap_length", "min"),
    )
    .sort_values(["mean_ci_overlap_norm", "mean_abs_prop_diff"], ascending=[True, False])
)

pairwise_summary

In [ ]:
imbalance_summary = (
    imbalance_all_df
    .groupby("outcome", as_index=False)
    .agg(
        mean_dominant_prop=("dominant_prop", "mean"),
        max_dominant_prop=("dominant_prop", "max"),
        mean_entropy_norm=("entropy_norm", "mean"),
        min_entropy_norm=("entropy_norm", "min"),
        mean_concentration=("concentration_index", "mean"),
        mean_l1_imbalance=("imbalance_l1_uniform", "mean"),
    )
    .sort_values(["mean_dominant_prop", "mean_entropy_norm"], ascending=[False, True])
)

imbalance_summary

In [ ]:
dominance_summary = (
    dominance_all_df
    .groupby(["outcome", "dominant_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_bins_dominant"})
    .sort_values(["outcome", "n_bins_dominant"], ascending=[True, False])
)

dominance_summary

# Overall

In [ ]:
tempbin_scorecard = (
    pairwise_summary
    .merge(imbalance_summary, on="outcome", how="outer")
)

# Example heuristic score:
# - reward large prop differences
# - reward low CI overlap
# - reward moderate/high dominance
# - reward lower entropy (cleaner bins)
tempbin_scorecard["tempbin_score"] = (
    0.35 * tempbin_scorecard["mean_abs_prop_diff"].fillna(0)
    - 0.30 * tempbin_scorecard["mean_ci_overlap_norm"].fillna(0)
    + 0.20 * tempbin_scorecard["mean_dominant_prop"].fillna(0)
    - 0.15 * tempbin_scorecard["mean_entropy_norm"].fillna(0)
)

tempbin_scorecard = tempbin_scorecard.sort_values("tempbin_score", ascending=False).reset_index(drop=True)
tempbin_scorecard
tempbin_scorecard.to_csv(CSV_DIR / "ALL_tempbin_scorecard.csv", index=False)

In [ ]:
overlap_fullsun = make_binned_overlap_table(
    df.query("sun == 'In full sun'"),
    value_col="<T-T_fixed+<T>>",
    outcome_col="thermal_comfort",
    bins=5,
    strategy="quantile"
)
plot_bin_overlap_with_errorbars(
    overlap,
    outcome_col="thermal_comfort",
    z=1.0,
    ylim=(0,0.6),
    title="Thermal comfort across temperature bins | full sun",
    output_path="figures/temp_overlap_fullsun_errorbars.png"
)

In [ ]:
for outcome in ["thermal_comfort","comfort3","comfort4_vc","comfort4_soft","comfort5_hard"]:
    ov = make_binned_overlap_table(df, "<T-T_fixed+<T>>", outcome, bins=5, strategy="quantile")
    ov.to_csv(f"csvs/temp_overlap_{outcome}.csv", index=False, encoding="utf-8-sig")

# Anem a probar el codi per comparar les particions

In [ ]:
# Example for one outcome
class_table_3g, overlap_3g, imbalance_3g, dominant_3g = summarize_temperature_bins(
    overlap,
    outcome_col="comfort3",   # replace with your chosen grouped outcome
    bin_col="bin"
)

class_table_4g, overlap_4g, imbalance_4g, dominant_4g = summarize_temperature_bins(
    df,
    outcome_col="comfort3_option1",   # replace with your chosen grouped outcome
    bin_col="T_bin"
)

# Optional: baseline comparison
class_table_soft, overlap_soft, imbalance_soft, dominant_soft = summarize_temperature_bins(
    df,
    outcome_col="comfort4_option2",
    bin_col="T_bin"
)